# Maia Lite 0.2.0 — Checkpoint Validation, Benchmark & Release Export

**Google Colab notebook · Post-SFT Stage 2 · No training**

This notebook verifies the completed `checkpoint-75000`, compares inference against a small multilingual smoke-test suite, merges the LoRA adapter with the original GPT-2-style base model, verifies numerical equivalence, and exports a self-contained Hugging Face release.

**Important:** It never modifies the original base checkpoint or the Stage 2 LoRA checkpoint. Export is written to a separate directory. The qualitative tests below are **smoke tests**, not a scientifically valid benchmark. The optional held-out evaluation cell requires an independently prepared JSONL dataset. GGUF/Ollama conversion is **not** attempted here because compatibility must be verified independently.

**Recommended runtime:** Colab with GPU (A100 preferred); enough Drive space for a complete ~355M parameter model plus temporary files. Run cells in order.

## 1. Environment and dependencies

In [ ]:
import sys, subprocess
# Pin to a mutually compatible range rather than silently upgrading to an unknown future API.
packages = ["transformers>=4.48,<5", "peft>=0.14,<1", "accelerate>=1.2,<2", "safetensors>=0.5,<1", "sentencepiece", "psutil"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
print("Dependencies installed. If transformers/peft were already imported, restart runtime before proceeding.")

In [ ]:
import os, json, math, time, hashlib, shutil, gc, platform
from pathlib import Path
import torch
import transformers, peft
from google.colab import drive

drive.mount("/content/drive")
print("Python:", platform.python_version())
print("Torch:", torch.__version__, "Transformers:", transformers.__version__, "PEFT:", peft.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (slow)")
print("Available GPU memory (GB):", round(torch.cuda.mem_get_info()[0]/2**30,2) if torch.cuda.is_available() else "N/A")

## 2. Paths and safety checks

Edit these only if your Google Drive layout differs. `BASE_DIR` must contain the **full model weights**, not merely a LoRA adapter. The tokenizer must be exactly the tokenizer used for SFT.

In [ ]:
PROJECT = Path("/content/drive/MyDrive/Colab_LLMs")
BASE_DIR = PROJECT / "Maia_Lite_Stage2/checkpoints/full/checkpoint-213624"
ADAPTER_DIR = PROJECT / "Maia_Lite_Stage2/sft_runs/stage2_multidomain_v3/checkpoint-75000"
TOKENIZER_DIR = PROJECT / "Maia_Lite_v3/tokenizer"
RELEASE_DIR = PROJECT / "Maia_Lite_Stage2/releases/maia-lite-0.2.0-hf"
REPORT_DIR = PROJECT / "Maia_Lite_Stage2/reports/maia-lite-0.2.0"

# These are separate output paths; originals are never overwritten.
assert RELEASE_DIR.resolve() not in (BASE_DIR.resolve(), ADAPTER_DIR.resolve(), TOKENIZER_DIR.resolve())
REPORT_DIR.mkdir(parents=True, exist_ok=True)

for name, path in [("Base", BASE_DIR), ("LoRA", ADAPTER_DIR), ("Tokenizer", TOKENIZER_DIR)]:
    if not path.is_dir(): raise FileNotFoundError(f"{name} directory not found: {path}")
    print(f"{name}: {path}")

print("Base files:", sorted(p.name for p in BASE_DIR.iterdir())[:30])
print("Adapter files:", sorted(p.name for p in ADAPTER_DIR.iterdir())[:30])
print("Tokenizer files:", sorted(p.name for p in TOKENIZER_DIR.iterdir())[:30])

## 3. Checkpoint integrity and configuration audit

In [ ]:
from transformers import AutoConfig, AutoTokenizer
from peft import PeftConfig

def has_one(directory, names):
    return any((directory / name).is_file() for name in names)

assert has_one(BASE_DIR, ["model.safetensors", "pytorch_model.bin", "model.safetensors.index.json", "pytorch_model.bin.index.json"]), "Base model weights not found"
assert (ADAPTER_DIR / "adapter_config.json").is_file(), "LoRA config missing"
assert has_one(ADAPTER_DIR, ["adapter_model.safetensors", "adapter_model.bin"]), "LoRA weights missing"

base_config = AutoConfig.from_pretrained(BASE_DIR, trust_remote_code=False)
peft_config = PeftConfig.from_pretrained(ADAPTER_DIR)
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_DIR, use_fast=True, trust_remote_code=False)

print("Base model type:", base_config.model_type)
print("Base architecture:", getattr(base_config, "architectures", None))
print("Base vocab size:", base_config.vocab_size)
print("Tokenizer length:", len(tokenizer))
print("EOS:", tokenizer.eos_token_id, "PAD:", tokenizer.pad_token_id)
print("LoRA rank:", peft_config.r, "alpha:", peft_config.lora_alpha)
print("LoRA targets:", peft_config.target_modules)
print("LoRA task:", peft_config.task_type)

assert base_config.model_type == "gpt2", "Unexpected architecture: review loader before proceeding"
assert len(tokenizer) == base_config.vocab_size, "Tokenizer/model vocab mismatch"
assert tokenizer.eos_token_id == 2, "EOS differs from training configuration"
assert tokenizer.pad_token_id == 1, "PAD differs from training configuration"
assert peft_config.peft_type.value == "LORA", "Expected LoRA adapter"
assert peft_config.r == 16, "Unexpected LoRA rank; inspect training provenance"
print("PASS: checkpoint structure and expected training metadata")

## 4. Load original base + trained LoRA adapter

Inference is performed in evaluation mode. **Do not resize token embeddings or add special tokens**: that would change the trained vocabulary. To reduce the risk of silently masking incompatibilities, we load the model with the original architecture and inspect the result.

In [ ]:
from transformers import GPT2LMHeadModel
from peft import PeftModel

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
base = GPT2LMHeadModel.from_pretrained(BASE_DIR, torch_dtype=dtype, low_cpu_mem_usage=True)
assert base.get_input_embeddings().weight.shape[0] == len(tokenizer)
model = PeftModel.from_pretrained(base, ADAPTER_DIR, is_trainable=False)
model = model.to(device).eval()
print("Loaded LoRA model on", device)
print("Total parameters:", sum(p.numel() for p in model.parameters()))
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))
print("Output embedding shape:", tuple(model.get_output_embeddings().weight.shape))

## 5. Multilingual qualitative smoke tests

Prompts preserve the exact training-style delimiters `System:`, `User:`, `Assistant:`. No new special tokens are introduced. Save raw generations for inspection; do not interpret them as correctness scores.

In [ ]:
SMOKE_TESTS = [
    {"id":"pt_intro", "domain":"linguistics", "language":"pt", "question":"Olá! Apresente-se em duas frases."},
    {"id":"en_intro", "domain":"linguistics", "language":"en", "question":"Hello! Introduce yourself in two sentences."},
    {"id":"es_intro", "domain":"linguistics", "language":"es", "question":"¡Hola! Preséntate en dos frases."},
    {"id":"pt_reason", "domain":"linguistics", "language":"pt", "question":"Qual é a diferença entre correlação e causalidade?"},
    {"id":"math_1", "domain":"mathematics", "language":"en", "question":"Solve 3x + 7 = 22. Show the result."},
    {"id":"math_2", "domain":"mathematics", "language":"en", "question":"What is the derivative of x^3 + 2x?"},
    {"id":"code_1", "domain":"programming", "language":"en", "question":"Write a Python function that returns the factorial of a nonnegative integer."},
    {"id":"code_2", "domain":"programming", "language":"en", "question":"Write Python code to compute the mean of a list of numbers."},
]
SYSTEM_PROMPT = "You are Maia Lite, a helpful assistant."

def format_prompt(question, system=SYSTEM_PROMPT):
    return f"System: {system}\nUser: {question}\nAssistant:"

@torch.inference_mode()
def generate_answer(question, max_new_tokens=160, do_sample=False, temperature=0.7):
    prompt = format_prompt(question)
    encoded = tokenizer(prompt, return_tensors="pt", add_special_tokens=False)
    encoded = {k:v.to(device) for k,v in encoded.items()}
    max_ctx = getattr(model.config, "n_positions", 1024)
    assert encoded["input_ids"].shape[1] + max_new_tokens <= max_ctx, "Prompt exceeds context window"
    kwargs = dict(**encoded, max_new_tokens=max_new_tokens, do_sample=do_sample,
                  eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.pad_token_id,
                  repetition_penalty=1.1)
    if do_sample: kwargs["temperature"] = temperature
    output = model.generate(**kwargs)
    new_ids = output[0, encoded["input_ids"].shape[1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True).strip()

results=[]
for item in SMOKE_TESTS:
    start=time.time()
    answer=generate_answer(item["question"])
    result={**item,"answer":answer,"seconds":round(time.time()-start,2)}
    results.append(result)
    print(f"\n[{item['id']}] {item['question']}\n{answer}\n")

smoke_path=REPORT_DIR/"smoke_tests.json"
smoke_path.write_text(json.dumps(results, ensure_ascii=False, indent=2),encoding="utf-8")
print("Saved:",smoke_path)

## 6. Optional held-out response-only loss evaluation

Provide a JSONL file with objects of the form `{"system": "...", "user": "...", "assistant": "...", "domain": "...", "language": "..."}`. It must contain **previously unseen examples**. The loss masks system/user prompt tokens, matching response-only SFT. This is an auxiliary evaluation, not an automatic correctness grade.

**Note:** If your original held-out corpus has different fields, adapt the parser explicitly. The default path is intentionally unset; the cell safely skips evaluation.

In [ ]:
HELDOUT_JSONL = None  # Example: str(PROJECT / "Maia_Lite_Stage2/heldout.jsonl")
MAX_HELDOUT = 300

def response_only_loss(record):
    system=record.get("system",SYSTEM_PROMPT)
    question=record["user"]
    response=record["assistant"]
    prefix=format_prompt(question, system)
    prefix_ids=tokenizer(prefix, add_special_tokens=False)["input_ids"]
    answer_ids=tokenizer(response, add_special_tokens=False)["input_ids"]+[tokenizer.eos_token_id]
    context_limit=getattr(model.config,"n_positions",1024)
    if len(prefix_ids)+len(answer_ids)>context_limit:
        return None  # Skip rather than truncate labels invisibly
    ids=prefix_ids+answer_ids
    labels=[-100]*len(prefix_ids)+answer_ids
    inputs=torch.tensor([ids],device=device)
    targets=torch.tensor([labels],device=device)
    with torch.inference_mode():
        out=model(input_ids=inputs, labels=targets)
    return float(out.loss.item()), len(answer_ids)

if HELDOUT_JSONL is None:
    print("Skipped: set HELDOUT_JSONL to an independent JSONL file to evaluate.")
else:
    heldout_path=Path(HELDOUT_JSONL)
    assert heldout_path.is_file(), heldout_path
    records=[]
    with heldout_path.open(encoding="utf-8") as f:
        for line in f:
            if line.strip(): records.append(json.loads(line))
            if len(records)>=MAX_HELDOUT: break
    grouped={}; skipped=0
    for rec in records:
        if not all(k in rec for k in ("user","assistant")):
            raise ValueError("Expected user and assistant fields")
        val=response_only_loss(rec)
        if val is None: skipped+=1; continue
        loss, ntok=val
        key=(rec.get("domain","unknown"),rec.get("language","unknown"))
        grouped.setdefault(key,[]).append((loss,ntok))
    summary={}
    for key, vals in grouped.items():
        # Weight by predicted response tokens, not by number of examples.
        avg=sum(loss*n for loss,n in vals)/sum(n for _,n in vals)
        summary["/".join(key)]={"examples":len(vals),"response_tokens":sum(n for _,n in vals),"loss":avg,"perplexity":math.exp(avg)}
    report={"results":summary,"skipped_overlength":skipped,"evaluated_from":str(heldout_path)}
    (REPORT_DIR/"heldout_metrics.json").write_text(json.dumps(report,indent=2),encoding="utf-8")
    print(json.dumps(report,indent=2))

## 7. Merge LoRA into base model

The merge creates a standalone model in memory. **Do not run any additional training here.** We compare logits before and after merging on the same prompt, using the same device and dtype. Tiny numerical differences are expected.

In [ ]:
probe=tokenizer(format_prompt("What is 2 + 2?"),return_tensors="pt",add_special_tokens=False)
probe={k:v.to(device) for k,v in probe.items()}
with torch.inference_mode():
    logits_before=model(**probe).logits.float().cpu()

merged=model.merge_and_unload(safe_merge=True)
merged.eval()
with torch.inference_mode():
    logits_after=merged(**probe).logits.float().cpu()

max_abs_diff=(logits_before-logits_after).abs().max().item()
mean_abs_diff=(logits_before-logits_after).abs().mean().item()
print("Merge max absolute logit difference:",max_abs_diff)
print("Merge mean absolute logit difference:",mean_abs_diff)
assert torch.isfinite(logits_after).all(), "Non-finite logits after merge"
assert max_abs_diff < 0.15, "Merge discrepancy too large: stop and investigate"
print("PASS: merged model produces close logits")
del model, base
gc.collect()
if torch.cuda.is_available():torch.cuda.empty_cache()

## 8. Save standalone Hugging Face release

The release is written to a **temporary sibling directory** first and only moved into the final path when saved successfully. If a release already exists, the notebook stops instead of overwriting it. Save the original tokenizer and generation settings alongside model weights.

In [ ]:
from transformers import GenerationConfig

if RELEASE_DIR.exists():
    raise FileExistsError(f"Release already exists: {RELEASE_DIR}. Inspect it or choose a new output path; nothing overwritten.")
RELEASE_DIR.parent.mkdir(parents=True,exist_ok=True)
tmp_dir=RELEASE_DIR.parent/(RELEASE_DIR.name+".incomplete")
if tmp_dir.exists():
    raise FileExistsError(f"Previous incomplete export exists: {tmp_dir}. Inspect it before deleting.")
tmp_dir.mkdir(parents=True)
try:
    merged.save_pretrained(tmp_dir, safe_serialization=True, max_shard_size="2GB")
    tokenizer.save_pretrained(tmp_dir)
    generation_config=GenerationConfig(
        max_new_tokens=160, do_sample=False,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
        repetition_penalty=1.1,
    )
    generation_config.save_pretrained(tmp_dir)
    manifest={
        "model":"Maia Lite 0.2.0", "architecture":"GPT-2 causal LM",
        "base_checkpoint":str(BASE_DIR), "lora_checkpoint":str(ADAPTER_DIR),
        "tokenizer_source":str(TOKENIZER_DIR),
        "training_steps":75000, "training_epochs":2,
        "merge_max_abs_logit_diff":max_abs_diff,
        "merge_mean_abs_logit_diff":mean_abs_diff,
        "torch_version":torch.__version__, "transformers_version":transformers.__version__,
        "peft_version":peft.__version__,
        "limitations":"Requires held-out and qualitative validation. Ollama/GGUF compatibility unverified."
    }
    (tmp_dir/"release_manifest.json").write_text(json.dumps(manifest,indent=2),encoding="utf-8")
    tmp_dir.rename(RELEASE_DIR)
except Exception:
    print("Export incomplete. Inspect temporary directory:",tmp_dir)
    raise
print("Exported:",RELEASE_DIR)
print("Files:",[(p.name,round(p.stat().st_size/2**20,1)) for p in RELEASE_DIR.iterdir() if p.is_file()])

## 9. Verify reloading from exported release

Reload the standalone model and compare its logits against the in-memory merged model. This is a critical check before copying it to a server.

In [ ]:
reloaded=GPT2LMHeadModel.from_pretrained(RELEASE_DIR,torch_dtype=dtype,low_cpu_mem_usage=True).to(device).eval()
reloaded_tokenizer=AutoTokenizer.from_pretrained(RELEASE_DIR,use_fast=True,trust_remote_code=False)
assert len(reloaded_tokenizer)==len(tokenizer)
assert reloaded_tokenizer.eos_token_id==tokenizer.eos_token_id
assert reloaded_tokenizer.pad_token_id==tokenizer.pad_token_id
with torch.inference_mode():
    reference=merged(**probe).logits.float().cpu()
    reread=reloaded(**probe).logits.float().cpu()
reload_diff=(reference-reread).abs().max().item()
print("Reload max absolute logit difference:",reload_diff)
assert torch.isfinite(reread).all()
assert reload_diff<0.15, "Reload mismatch: stop and investigate"
print("PASS: standalone release reloads successfully")
del reloaded, merged
gc.collect()
if torch.cuda.is_available():torch.cuda.empty_cache()

## 10. Generate checksums and release notes

SHA-256 hashes make it possible to verify a subsequent download or deployment. They do not prove model quality.

In [ ]:
def sha256_file(path,chunk_size=8*1024*1024):
    digest=hashlib.sha256()
    with open(path,"rb") as stream:
        while chunk:=stream.read(chunk_size):digest.update(chunk)
    return digest.hexdigest()

hashes={p.name:sha256_file(p) for p in sorted(RELEASE_DIR.iterdir()) if p.is_file()}
(RELEASE_DIR/"SHA256SUMS.json").write_text(json.dumps(hashes,indent=2),encoding="utf-8")
readme="""# Maia Lite 0.2.0 (Hugging Face export)

GPT-2-style causal language model (~355M base parameters), fine-tuned with a Stage 2 multilingual/multidomain LoRA adapter, merged into the original base weights.

## Prompt format
```
System: You are Maia Lite, a helpful assistant.
User: Hello!
Assistant:
```

Use the supplied tokenizer. EOS token ID: 2; PAD token ID: 1. Maximum context length: check `config.json` (`n_positions`, expected 1024).

## Status
Training completed: 75,000 optimizer steps, 2 epochs. Training loss alone does not establish quality. Inspect qualitative smoke tests and held-out evaluations before publishing performance claims.

## Deployment
Load using `transformers.GPT2LMHeadModel.from_pretrained(...)` and the bundled tokenizer. GGUF/Ollama compatibility has **not** been established; do not assume `ollama create` supports this export.

## Provenance
See `release_manifest.json`, `SHA256SUMS.json`, and the independent report directory.
"""
(RELEASE_DIR/"README.md").write_text(readme,encoding="utf-8")
print("Release directory:",RELEASE_DIR)
print("Files hashed:",len(hashes))
print("Reports:",REPORT_DIR)
print("DONE — model preserved in HF format. No GGUF conversion attempted.")

## 11. Recommended next steps

1. Review `smoke_tests.json` for repetitive answers, multilingual behavior, mathematical errors and code quality.
2. Evaluate **held-out** test examples from each language/domain, with no overlap with training or model-selection examples.
3. Compare the merged model against the original pretrained base and earlier Stage 2 checkpoints, using the same prompts and decoding settings.
4. Only after verifying GGUF conversion and Ollama architecture/tokenizer support, plan the Maia Chat deployment. Alternatively, expose the HF model via a dedicated Python inference service connected to Maia Chat.
5. Archive the original `checkpoint-75000` LoRA adapter, base checkpoint, tokenizer, release manifest, and checksums. Preserve them separately for reproducibility and future Maia Lite 0.3.0 training.

**Caution:** A low response-only training loss is not evidence that the model reliably solves mathematics or writes correct Python. Automated executable validation and blinded benchmark questions are required.